## Getting Started: Prepping MIMIC4Notes

In [ ]:
#!/usr/bin/env python3
import argparse
import json
import os
import torch
import traceback
import numpy as np
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path
from tqdm import tqdm

# Set correct directory pathing
import os
import sys
# current_dir = os.path.dirname(os.path.abspath(__file__))
# parent_dir = os.path.dirname(current_dir)
# sys.path.insert(0, parent_dir)

# Import project modules
# from rdma.rdrag.entity import LLMRDExtractor, BaseRDExtractor, RetrievalEnhancedRDExtractor,MultiIterativeRDExtractor,IterativeLLMRDExtractor
# from rdma.utils.embedding import EmbeddingsManager
# from rdma.hporag.context import ContextExtractor
# from rdma.utils.llm_client import LocalLLMClient, APILLMClient
# from rdma.utils.setup import setup_device

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [ ]:
SAMPLE_SIZE = 5

**Quick Notes**

In `/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/base_dataset.py`, you need to change in `schema_reader` and `csv_reader`:
```
parse_options=pv.ParseOptions(delimiter=delimiter,
                                              newlines_in_values=True)
```

### Load MIMIC notes and return sampled notes in JSON

In [ ]:
from pyhealth.datasets.mimic4 import MIMIC4NoteDataset

In [ ]:
NOTE_ROOT = '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp'

dataset = MIMIC4NoteDataset(root=NOTE_ROOT, tables=["discharge"])
note_df = dataset.global_event_df.collect().to_pandas()

In [ ]:
patient_notes = note_df.groupby("patient_id")['discharge/text'].apply(lambda texts: "\n\n".join(texts))
filtered_notes = patient_notes[patient_notes.str.len() > 500]
samples = filtered_notes.sample(n=SAMPLE_SIZE, random_state=42)

In [ ]:
sampled_data = {
        str(pid): {
            "clinical_text": text,
            "patient_id": str(pid),
            "hadm_id": "",
            "category": "",
            "chartdate": ""
        }
        for pid, text in samples.items()
    }

### Print and store sampled_data

In [ ]:
print(sampled_data['13106750']['chartdate'])

In [ ]:
%store sampled_data